In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
from functions import get_experiment_data


import numpy as np
import pandas as pd

def hivstatus_classification_auc(X, y, label=""):
    X = np.array(X)
    y = np.array(y)
    
    # NaN 제거
    valid_mask = ~pd.isna(y)
    X = X[valid_mask]
    y = y[valid_mask]
    
    # label encoding
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # 5-fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    
    for train_idx, test_idx in skf.split(X, y_encoded):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
        
        clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)
        clf.fit(X_train, y_train)
        y_prob = clf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        aucs.append(auc)
    
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    
    print(f"\n{'='*50}")
    print(f" {label}")
    print(f"{'='*50}")
    print(f"Mean AUC: {mean_auc:.4f} ± {std_auc:.4f}")
    print(f"(높을수록 biological signal 보존이 잘 된 것)")
    
    return {"label": label, "mean_auc": mean_auc, "std_auc": std_auc}

def study_classification_auc(X, batch, label=""):
    X = np.array(X)
    batch = np.array(batch)
    
    le = LabelEncoder()
    batch_encoded = le.fit_transform(batch)
    n_classes = len(le.classes_)
    
    # 5-fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    
    for train_idx, test_idx in skf.split(X, batch_encoded):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = batch_encoded[train_idx], batch_encoded[test_idx]
        
        clf = OneVsRestClassifier(
            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)
        )
        y_train_bin = label_binarize(y_train, classes=range(n_classes))
        clf.fit(X_train, y_train_bin)
        
        y_test_bin = label_binarize(y_test, classes=range(n_classes))
        y_prob = clf.predict_proba(X_test)
        
        auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
        aucs.append(auc)
    
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    
    print(f"\n{'='*50}")
    print(f" {label}")
    print(f"{'='*50}")
    print(f"Mean AUC: {mean_auc:.4f} ± {std_auc:.4f}")
    print(f"(낮을수록 batch correction이 잘 된 것)")
    
    return {"label": label, "mean_auc": mean_auc, "std_auc": std_auc}


In [ ]:

X_raw, meta_raw, batch_raw, _ = get_experiment_data(
    design_id="df2" ,
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df2
Design name             : minproc_otu_count
Description             : Minimally processed HIVRC at the OTU level, raw count
Aggregation             : None
Normalize               : False
Cleanset Filtering      : False
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (1032, 38125)
meta_data               : (1032, 11)
n_batches               : 17


In [ ]:
# study별 HIV+/HIV- 비율
meta_raw.groupby('Study')['hivstatus'].value_counts(normalize=True).unstack(fill_value=0)


hivstatus,0,1
Study,,
Dillon,0.437500,0.562500
Dinh,0.416667,0.583333
Dubourg,0.446429,0.553571
Lozupone 2013,0.351351,0.648649
Monaco,0.349057,0.650943
Mutlu,0.525000,0.475000
Noguera-Julian,0.145923,0.854077
Nowak2015,0.225000,0.775000
Nowak2017,0.428571,0.571429


In [ ]:
balanced = ["Mutlu", "Dillon", "Dinh", "Dubourg", "Nowak2017"]
subset = meta[meta['Study'].isin(balanced)]
print(f"총 샘플 수: {len(subset)}")
print(subset.groupby('Study')['hivstatus'].value_counts())

총 샘플 수: 283
Study      hivstatus
Dillon     1            18
           0            14
Dinh       1            21
           0            15
Dubourg    1            31
           0            25
Mutlu      0            21
           1            19
Nowak2017  1            68
           0            51
Name: count, dtype: int64


In [2]:
import pandas as pd

# balanced studies 필터링
BALANCED_STUDIES = ["Mutlu", "Dillon", "Dinh", "Dubourg", "Nowak2017"]

# df4 기준

X_filtered_df4 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_filtered_cutoff0.005.csv", index_col=0)


meta_filtered_df4 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df4_filtered_cutoff0.005.csv", index_col=0)
batch_filtered_df4 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df4_filtered_cutoff0.005.csv", index_col=0)

X_df4_latgentgee =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df4/phase2/X_corrected_df4_clean_latentgee.csv", index_col=0)

X_df4_combat =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_corrected_combat.csv", index_col=0)
X_df4_mmuphine =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_corrected_MMUPHin.csv", index_col=0)
X_df4_conqur =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_corrected_ConQuR.csv", index_col=0)

balanced_mask = batch_filtered_df4['Study'].isin(BALANCED_STUDIES).values

X_balanced = X_filtered_df4.values[balanced_mask]
X_balanced_latentgee = X_df4_latgentgee.values[balanced_mask]
X_balanced_combat = X_df4_combat.values[balanced_mask]
X_balanced_mmuphin = X_df4_mmuphine.values[balanced_mask]
X_balanced_conqur = X_df4_conqur.values[balanced_mask]

bio_balanced = meta_filtered_df4['hivstatus'].values[balanced_mask]
batch_balanced = batch_filtered_df4['Study'].values[balanced_mask]

print(f"Balanced subset 샘플 수: {balanced_mask.sum()}")
print(pd.Series(batch_balanced).value_counts())

Balanced subset 샘플 수: 243
Nowak2017    119
Dubourg       56
Dinh          36
Dillon        32
Name: count, dtype: int64


In [ ]:


# 1. hivstatus classification (biological signal 보존)
r_before    = hivstatus_classification_auc(X_balanced, bio_balanced, "balanced — Before")
r_latentgee = hivstatus_classification_auc(X_balanced_latentgee, bio_balanced, "balanced — LatentGEE")
r_combat    = hivstatus_classification_auc(X_balanced_combat, bio_balanced, "balanced — ComBat")
r_mmuphin   = hivstatus_classification_auc(X_balanced_mmuphin, bio_balanced, "balanced — MMUPHin")
r_conqur    = hivstatus_classification_auc(X_balanced_conqur, bio_balanced, "balanced — ConQuR")



 balanced — Before
Mean AUC: 0.6486 ± 0.0519
(높을수록 biological signal 보존이 잘 된 것)

 balanced — LatentGEE
Mean AUC: 0.5478 ± 0.0617
(높을수록 biological signal 보존이 잘 된 것)

 balanced — ComBat
Mean AUC: 0.6946 ± 0.0485
(높을수록 biological signal 보존이 잘 된 것)

 balanced — MMUPHin
Mean AUC: 0.7086 ± 0.0291
(높을수록 biological signal 보존이 잘 된 것)

 balanced — ConQuR
Mean AUC: 1.0000 ± 0.0000
(높을수록 biological signal 보존이 잘 된 것)


NameError: name 'OneVsRestClassifier' is not defined

In [ ]:


# 2. study classification (batch effect 제거)
s_before    = study_classification_auc(X_balanced, batch_balanced, "balanced — Before")
s_latentgee = study_classification_auc(X_balanced_latentgee, batch_balanced, "balanced — LatentGEE")
s_combat    = study_classification_auc(X_balanced_combat, batch_balanced, "balanced — ComBat")
s_mmuphin   = study_classification_auc(X_balanced_mmuphin, batch_balanced, "balanced — MMUPHin")
s_conqur    = study_classification_auc(X_balanced_conqur, batch_balanced, "balanced — ConQuR")



 balanced — Before
Mean AUC: 1.0000 ± 0.0000
(낮을수록 batch correction이 잘 된 것)

 balanced — LatentGEE
Mean AUC: 0.6263 ± 0.0434
(낮을수록 batch correction이 잘 된 것)

 balanced — ComBat
Mean AUC: 1.0000 ± 0.0000
(낮을수록 batch correction이 잘 된 것)

 balanced — MMUPHin
Mean AUC: 0.9993 ± 0.0014
(낮을수록 batch correction이 잘 된 것)

 balanced — ConQuR
Mean AUC: 1.0000 ± 0.0000
(낮을수록 batch correction이 잘 된 것)


In [6]:
# df5 데이터 준비
X_filtered_df5 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df5_filtered_cutoff0.05.csv", index_col=0)
meta_filtered_df5 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df5_filtered_cutoff0.05.csv", index_col=0)
batch_filtered_df5 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df5_filtered_cutoff0.05.csv", index_col=0)

# corrected 데이터 로드
X_corrected_df5 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df5/phase2/df5_X_corrected_trial358_cutoff0.05_2026-05-27.csv",
    index_col=0
)
X_corrected_df5_clean =  pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df5/phase2/X_corrected_df5_trial358_clean_latentgee.csv",
    index_col=0
)

# ComBat, MMUPHin, ConQuR corrected 로드
X_corrected_combat_df5 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df5_corrected_combat.csv", index_col=0)
X_corrected_mmuphin_df5 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df5_corrected_MMUPHin.csv", index_col=0)
X_corrected_conqur_df5 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df5_corrected_ConQuR.csv", index_col=0)

bio_labels_df5 = meta_filtered_df5['hivstatus'].values
batch_values_df5 = batch_filtered_df5.values if isinstance(batch_filtered_df5, pd.Series) else batch_filtered_df5['Study'].values

# study classification
print("=== Study Classification AUC (낮을수록 좋음) ===")
study_classification_auc(X_filtered_df5.values, batch_values_df5, "df5 — Before")
study_classification_auc(X_corrected_df5_clean.values, batch_values_df5, "df5 — LatentGEE")
study_classification_auc(X_corrected_combat_df5.values, batch_values_df5, "df5 — ComBat")
study_classification_auc(X_corrected_mmuphin_df5.values, batch_values_df5, "df5 — MMUPHin")
study_classification_auc(X_corrected_conqur_df5.values, batch_values_df5, "df5 — ConQuR")

# hivstatus classification
print("\n=== hivstatus Classification AUC (높을수록 좋음) ===")
hivstatus_classification_auc(X_filtered_df5.values, bio_labels_df5, "df5 — Before")
hivstatus_classification_auc(X_corrected_df5_clean.values, bio_labels_df5, "df5 — LatentGEE")
hivstatus_classification_auc(X_corrected_combat_df5.values, bio_labels_df5, "df5 — ComBat")
hivstatus_classification_auc(X_corrected_mmuphin_df5.values, bio_labels_df5, "df5 — MMUPHin")
hivstatus_classification_auc(X_corrected_conqur_df5.values, bio_labels_df5, "df5 — ConQuR")

=== Study Classification AUC (낮을수록 좋음) ===

 df5 — Before
Mean AUC: 1.0000 ± 0.0000
(낮을수록 batch correction이 잘 된 것)

 df5 — LatentGEE
Mean AUC: 0.6106 ± 0.0370
(낮을수록 batch correction이 잘 된 것)

 df5 — ComBat
Mean AUC: 1.0000 ± 0.0000
(낮을수록 batch correction이 잘 된 것)

 df5 — MMUPHin
Mean AUC: 0.9990 ± 0.0011
(낮을수록 batch correction이 잘 된 것)

 df5 — ConQuR
Mean AUC: 0.9990 ± 0.0009
(낮을수록 batch correction이 잘 된 것)

=== hivstatus Classification AUC (높을수록 좋음) ===

 df5 — Before
Mean AUC: 0.6813 ± 0.0346
(높을수록 biological signal 보존이 잘 된 것)

 df5 — LatentGEE
Mean AUC: 0.5083 ± 0.0882
(높을수록 biological signal 보존이 잘 된 것)

 df5 — ComBat
Mean AUC: 0.7004 ± 0.0320
(높을수록 biological signal 보존이 잘 된 것)

 df5 — MMUPHin
Mean AUC: 0.6835 ± 0.0439
(높을수록 biological signal 보존이 잘 된 것)

 df5 — ConQuR
Mean AUC: 0.7550 ± 0.0235
(높을수록 biological signal 보존이 잘 된 것)


{'label': 'df5 — ConQuR',
 'mean_auc': 0.7550138573948098,
 'std_auc': 0.023500318800111504}

In [7]:
# df6 데이터 준비
X_filtered_df6 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df6_filtered_cutoff0.1.csv", index_col=0)
meta_filtered_df6 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df6_filtered_cutoff0.1.csv", index_col=0)
batch_filtered_df6 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df6_filtered_cutoff0.1.csv", index_col=0)

# corrected 데이터 로드
X_corrected_df6 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df6/phase2/df6_X_corrected_trial1244_cutoff0.1_2026-05-20.csv",
    index_col=0
)
X_corrected_df6_clean =  pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df6/phase2/X_corrected_df6_clean_latentgee.csv",
    index_col=0
)

# ComBat, MMUPHin, ConQuR corrected 로드
X_corrected_combat_df6 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df6_corrected_combat.csv", index_col=0)
X_corrected_mmuphin_df6 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df6_corrected_MMUPHin.csv", index_col=0)
X_corrected_conqur_df6 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df6_corrected_ConQuR.csv", index_col=0)

bio_labels_df6 = meta_filtered_df6['hivstatus'].values
batch_values_df6 = batch_filtered_df6.values if isinstance(batch_filtered_df6, pd.Series) else batch_filtered_df6['Study'].values

# study classification
print("=== Study Classification AUC (낮을수록 좋음) ===")
study_classification_auc(X_filtered_df6.values, batch_values_df6, "df6 — Before")
study_classification_auc(X_corrected_df6_clean.values, batch_values_df6, "df6 — LatentGEE")
study_classification_auc(X_corrected_combat_df6.values, batch_values_df6, "df6 — ComBat")
study_classification_auc(X_corrected_mmuphin_df6.values, batch_values_df6, "df6 — MMUPHin")
study_classification_auc(X_corrected_conqur_df6.values, batch_values_df6, "df6 — ConQuR")

# hivstatus classification
print("\n=== hivstatus Classification AUC (높을수록 좋음) ===")
hivstatus_classification_auc(X_filtered_df6.values, bio_labels_df6, "df6 — Before")
hivstatus_classification_auc(X_corrected_df6_clean.values, bio_labels_df6, "df6 — LatentGEE")
hivstatus_classification_auc(X_corrected_combat_df6.values, bio_labels_df6, "df6 — ComBat")
hivstatus_classification_auc(X_corrected_mmuphin_df6.values, bio_labels_df6, "df6 — MMUPHin")
hivstatus_classification_auc(X_corrected_conqur_df6.values, bio_labels_df6, "df6 — ConQuR")

=== Study Classification AUC (낮을수록 좋음) ===

 df6 — Before
Mean AUC: 0.9999 ± 0.0002
(낮을수록 batch correction이 잘 된 것)

 df6 — LatentGEE
Mean AUC: 0.9034 ± 0.0583
(낮을수록 batch correction이 잘 된 것)

 df6 — ComBat
Mean AUC: 1.0000 ± 0.0000
(낮을수록 batch correction이 잘 된 것)

 df6 — MMUPHin
Mean AUC: 0.9996 ± 0.0007
(낮을수록 batch correction이 잘 된 것)

 df6 — ConQuR
Mean AUC: 0.9999 ± 0.0002
(낮을수록 batch correction이 잘 된 것)

=== hivstatus Classification AUC (높을수록 좋음) ===

 df6 — Before
Mean AUC: 0.7141 ± 0.0616
(높을수록 biological signal 보존이 잘 된 것)

 df6 — LatentGEE
Mean AUC: 0.6080 ± 0.0418
(높을수록 biological signal 보존이 잘 된 것)

 df6 — ComBat
Mean AUC: 0.7006 ± 0.0391
(높을수록 biological signal 보존이 잘 된 것)

 df6 — MMUPHin
Mean AUC: 0.6794 ± 0.0284
(높을수록 biological signal 보존이 잘 된 것)

 df6 — ConQuR
Mean AUC: 0.8783 ± 0.0390
(높을수록 biological signal 보존이 잘 된 것)


{'label': 'df6 — ConQuR',
 'mean_auc': 0.8782942806752331,
 'std_auc': 0.03898059004233846}